In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import pandas as pd

from torch.backends import cudnn
from tqdm import tqdm

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("..")
from pneuma_seeker.core.ir_system.main import IRSystem
from pneuma_seeker.core.ir_system.data_model import RetrieverType
from pneuma_seeker.utils.config import Config
from pneuma_seeker.utils.logger import setup_logger
from pneuma_seeker.core.ir_system.data_model import AbstractDocument
from pneuma_seeker.core.ir_system.data_model import Table, TableContext
from pneuma_seeker.model.interface.model_factory import get_embed_model, get_llm

/home/pneuma_admin/miniconda3/envs/pneuma/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
INDEXING_ARCHEOLOGY = False
INDEXING_BIOMEDICAL = False
INDEXING_ENVIRONMENT = False
INDEXING_TAG = False
INDEXING_BUYSITE = True

In [3]:
config = Config("../.env")
llm_path = "model/weight/qwen3-4b-instruct"
embed_model_path = "model/weight/bge-base"
logger = setup_logger(log_path=os.path.join(".", "log"))

In [4]:
LARGE_BUYSITE_DATASET = [
    "JI_ADDRESS",
    "JI_ITEM",
    "JI_PURCHASE_ORDER_AUDIT_TRAIL",
    "JI_PURCHASE_ORDER_CUSTOM_FIELDS_GROUP_RESPONSE_16366401",
    "JI_PURCHASE_ORDER_CUSTOM_FIELDS_SINGLE_VALUE_RESPONSE",
    "JI_PURCHASE_ORDER_CUSTOM_FIELDS",
    "JI_PURCHASE_ORDER_LINE",
    "JI_PURCHASE_ORDER_WORKFLOW",
    "JI_PURCHASE_ORDER",
    "JI_REQUISITION_AUDIT_TRAIL",
    "JI_REQUISITION_CUSTOM_FIELDS_GROUP_RESPONSE_16366401",
    "JI_REQUISITION",   
]

In [5]:
def index_dataset(dataset_name: str, metadata_available = False):
    DATASET_DIR = f"../../data_src/{dataset_name}/dataset"
    documents: list[AbstractDocument] = []
    dataset = os.listdir(DATASET_DIR)
    for table_name in tqdm(dataset, desc="Loading dataset..."):
        try:
            if dataset_name == "buysite" and table_name[:-4] in LARGE_BUYSITE_DATASET:
                table = pd.read_csv(f"{DATASET_DIR}/{table_name}", nrows=1000)
            else:
                table = pd.read_csv(f"{DATASET_DIR}/{table_name}")
        except:
            continue
        documents.append(
            Table(
                doc_id=f"{DATASET_DIR}/{table_name}",
                retriever_type=RetrieverType.PNEUMA_RETRIEVER,
                content=table,
                metadata={
                    "table_name": f"{DATASET_DIR}/{table_name}",
                    "dataset_name": dataset_name
                }
            )
        )
    
    if metadata_available:
        dataset_metadata = pd.read_csv(f"../../data_src/{dataset_name}/metadata.csv")
        for _, row in tqdm(dataset_metadata.iterrows(), desc="Loading metadata..."):
            table_name = row["table_name"]
            description = row["description"]
            documents.append(
                TableContext(
                    doc_id=f"context_{DATASET_DIR}/{table_name}",
                    retriever_type=RetrieverType.PNEUMA_RETRIEVER,
                    content=description,
                    metadata={
                        "table_name": f"{DATASET_DIR}/{table_name}",
                        "dataset_name": dataset_name,
                        "type": "description",
                    }
                )
            )

    ir_sys = IRSystem(
        get_llm(llm_path, config)(llm_path, config, logger),
        get_embed_model()(embed_model_path, config, logger),
        logger,
        config,
    )
    ir_sys.index_documents(
        RetrieverType.PNEUMA_RETRIEVER,
        documents
    )

In [ ]:
if INDEXING_ARCHEOLOGY:
    index_dataset("archeology", True)
if INDEXING_BIOMEDICAL:
    index_dataset("biomedical", True)
if INDEXING_ENVIRONMENT:
    index_dataset("environment", True)
if INDEXING_TAG:
    index_dataset("tag", True)
if INDEXING_BUYSITE:
    index_dataset("buysite", True)

Loading dataset...: 100%|██████████| 42/42 [00:00<00:00, 121.55it/s]
Loading metadata...: 42it [00:00, 24593.15it/s]

[2025-11-14 13:49:48] INFO in logger: [IR System] Indexing documents on the retriever RetrieverType.PNEUMA_RETRIEVER.


Enter indexing...
About to summarize and sample rows...


Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.09it/s]
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [ ]:
# Extra: Processing for TAG data
# for topic in os.listdir(DATASET_DIR):
#     topic_path = f"dataset/{topic}"
#     if os.path.isdir(topic_path):
#         for table_fname in os.listdir(topic_path):
#             original_table_path = f"{topic_path}/{table_fname}"
#             appended_table_path = f"{topic_path}/{topic}_{table_fname}"
#             print(f"=> {original_table_path} => {appended_table_path}")
#             os.rename(original_table_path, appended_table_path)
# Future-TODO: don't forget to move the tables outside (manually for now)